In [1]:
import os
import pickle 
from tqdm.auto import tqdm
import pandas as pd
import torch
# from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph
from Nodes.States.states import  OverallState,InputState,OutputState
from Configuration.configurations import ConfigSchema
from Nodes.schema_retriever import schema_retriever
# from Nodes.schema_filtering import schema_filtering
from Nodes.cypher_generator import cypher_generator
from Nodes.cypher_executor import cypher_executor

# from neo4j import GraphDatabase,basic_auth
# from neo4j_graphrag.schema import get_structured_schema, format_schema,  get_schema


In [2]:
#Define the graph
workflow = StateGraph(OverallState, input=InputState, output=OutputState, config_schema=ConfigSchema)
#Add nodes
workflow.add_node(schema_retriever)
workflow.add_node(cypher_generator)
workflow.add_node(cypher_executor)
#Add edges
workflow.add_edge(START, "schema_retriever")
workflow.add_edge("schema_retriever", "cypher_generator")
workflow.add_edge("cypher_generator", "cypher_executor")    
workflow.add_edge("cypher_executor",  END)

app = workflow.compile()
# display(Image(app.get_graph().draw_mermaid_png(output_file_path="Images/graph.png")))

In [ ]:
# import subprocess
# import torch
# from tqdm.auto import tqdm


# def get_gpu_memory_in_gb():
#     result = subprocess.run(['nvidia-smi', 
#                              '--query-gpu=memory.total,memory.used,memory.free', 
#                              '--format=csv,noheader,nounits'],
#                             stdout=subprocess.PIPE, text=True)
#     memory_info = result.stdout.strip().split('\n')
#     for i, gpu in enumerate(memory_info):
#         total, used, free = map(int, gpu.split(','))
#         print(f"GPU {i}: Total: {total / 1024:.2f} GB, Used: {used / 1024:.2f} GB, Free: {free / 1024:.2f} GB")

# get_gpu_memory_in_gb()

In [ ]:
# """
# For one question at a time
# """
# from Configuration.models import models #import models to evaluate

# model_name = 'llama3_2_3b_local'
# model = models[model_name]()

# inputs = {
#     'question':'Find the `customerID` and `shipCity` for the customer with the highest total order value (sum of `unitPrice` * `quantity` for all their orders).',
#     'database_url':'neo4j+s://demo.neo4jlabs.com',
#     'database_name':'northwind',
#     'database_user':'northwind',
#     'database_password':'northwind',
#     # 'loop_count':0
# }
# config = {'configurable': {'model': model, 'model_name':model_name}}

# app.invoke(inputs, config =config)

In [ ]:
# """
# For one question at a time, reading from excel
# """
# from Configuration.models import models #import models to evaluate
# model_name = 'llama3_2_3b_local'
# model = models[model_name]()

# # df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
# df = pd.read_excel('../data/text2cypher-sustaingraph/sustaintest.xlsx')

# # row = df.iloc[27] #offshoreleaks example
# row = df.iloc[549]

# # inputs = {
# #     'question': row['question'],
# #     'database_url': row['url'],
# #     'database_name': row['database_name'],
# #     'database_user': row['username'],
# #     'database_password': row['password'],
# #     # 'loop_count': row['loop_count']  
# # }

# inputs = {
#     'question': row['Question'],
#     'database_url': os.getenv('database_url'),
#     'database_name': os.getenv('database_name'),
#     'database_user': os.getenv('database_user'),
#     'database_password': os.getenv('database_password')
#     # 'loop_count': row['loop_count']  
# }

# print(inputs)

# config = {'configurable': {'model': model, 'model_name':model_name}}

# app.invoke(inputs, config =config)
# get_gpu_memory_in_gb()

## RUN SEQUENTIALLY

In [ ]:
# def evaluate_model (model_name, model, dataset):
#     ''' 
#     Evaluate the selected model on the given dataset, by invoking the graph workflow.
#     Save the dataframe with the questions and the generated cypher queries to the Results folder.
#     Return the dataframe
#     '''
#     # Get datasets 
#     if dataset =='sustain':
#         df = pd.read_excel('../data/text2cypher-sustaingraph/LLM Cypher Set.xlsx',sheet_name='Queries')
#         subfolder = 'SustainDataset'
#     elif dataset == 'benchmark2025':
#         df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
#         #update df only with rows with a database access
#         df = df[df['url'].notna()].reset_index(drop=True)
#         subfolder = 'BenchmarkDataset2025'
#     elif dataset == 'benchmark2024':
#         df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
#         #update df only with rows with a database access
#         df = df[df['url'].notna()].reset_index(drop=True)
#         subfolder = 'BenchmarkDataset2024'
#     # Run model on dataset
#     df_results = df.copy() # Copy of the df with questions and expected answers, to add the answers of the llm 
#     config = {'configurable': {'model': model, 'model_name':model_name}}
#     print(f'Starting evaluation for {model_name} on {dataset} ...')
#     for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
#         question = row['question']
        
#         # Get credentials
#         required_cols = {'database_name', 'url', 'user', 'password'}
#         if required_cols.issubset(df_results.columns):
#             database_url = row['url']
#             database_name = row['database_name']
#             database_user = row['user']
#             database_password = row['password']
#         else:
#             #its the sustain dataset
#             database_url = os.getenv('database_url')
#             database_name = os.getenv('database_name')
#             database_user = os.getenv('database_user')
#             database_password = os.getenv('database_password')
        
#         # Configuration 
#         inputs = {
#             'question':question,
#             'database_url': database_url,
#             'database_name':database_name,
#             'database_user':database_user,
#             'database_password':database_password
#         }
#         torch.cuda.empty_cache() #clear gpu cache
#         # Run the graph
#         result =app.invoke(inputs, config =config) #,debug=False
#         df_results.at[i, 'Generated_Cypher'] = result['cypher_statement']
#         df_results.at[i, 'Output'] = str(result['database_records'])
#         torch.cuda.empty_cache()#clear gpu cache
#     # Save results
#     output_dir = f"Results/{subfolder}/{model_name}/"
#     # Create the directory if it doesn't exist
#     os.makedirs(output_dir, exist_ok=True)
#     # Save results
#     df_results.to_pickle(os.path.join(output_dir, f"{model_name}.pkl"))
#     df_results.to_excel(os.path.join(output_dir, f"{model_name}.xlsx"), index=False)# Set index=False to exclude the index column in Excel
#     return df_results

In [ ]:
def evaluate_model_and_save (model_name, model, dataset):
    ''' 
    Evaluate the selected model on the given dataset, by invoking the graph workflow.
    Save the dataframe with the questions and the generated cypher queries to the Results folder.
    Return the dataframe
    '''
    # Get datasets 
    # Get datasets 
    if dataset =='sustain':
        df = pd.read_excel('../data/text2cypher-sustaingraph/sustaintest.xlsx',sheet_name='Queries')
        subfolder = 'SustainDataset'
    elif dataset == 'benchmark2025':
        df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2025'
    elif dataset == 'benchmark2024':
        df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2024'

    output_dir = f"Results/{subfolder}/{model_name}_local/"
    # Create the directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    results_path_pkl = os.path.join(output_dir, f"{model_name}_local.pkl")
    results_path_xlsx = os.path.join(output_dir, f"{model_name}_local.xlsx")
    print(results_path_xlsx)

    if os.path.exists(results_path_xlsx):
        print("Result Excel file exists")
        # Read Excel file
        df_results = pd.read_excel(results_path_xlsx)
        # print(df_results)
        # Save it as pickle
        df_results.to_pickle(results_path_pkl)
        print("Converted Excel to pickle")
        # Read the pickle
        df_results = pd.read_pickle(results_path_pkl)
        print("Read the pickle")
    
    else:
        print("Making result file")
        # Run model on dataset
        df_results = df.copy() # Copy of the df with questions and expected answers, to add the answers of the llm 

    config = {'configurable': {'model': model, 'model_name':model_name}}
    print(f'Starting evaluation for {model_name} on {dataset} ...')

    for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
        if pd.isna(row['Generated_Cypher']): 
            if dataset == 'sustain':
                question = row['Question'] #FIX: for benchmark2025, it's question, not Question
            else: #dataset is benchmark
                question = row['question']
                database_name = row['database_name']         

            # Get credentials
            required_cols = {'database_name', 'url', 'username', 'password'}
            if required_cols.issubset(df_results.columns):
                database_url = row['url']
                database_name = row['database_name']
                database_user = row['username']
                database_password = row['password']
            else:
                #its the sustain dataset
                database_url = os.getenv('database_url')
                database_name = os.getenv('database_name')
                database_user = os.getenv('database_user')
                database_password = os.getenv('database_password')
            
            # Configuration 
            inputs = {
                'question':question,
                'database_url': database_url,
                'database_name':database_name,
                'database_user':database_user,
                'database_password':database_password
            }
            torch.cuda.empty_cache() #clear gpu cache
            # Run the graph
            result =app.invoke(inputs, config =config) #,debug=False
            # get_gpu_memory_in_gb()
            df_results.at[i, 'Generated_Cypher'] = result['cypher_statement']
            df_results.at[i, 'Output'] = str(result['database_records'])
            torch.cuda.empty_cache()#clear gpu cache

            df_results.to_pickle(results_path_pkl)
            df_results.to_excel(results_path_xlsx, index=False)
        else:
            print(f"Skipping row {i}, already computed.")

    return df_results

In [ ]:
# def evaluate_model_for_hf (model_name, model, dataset):
#     ''' 
#     Evaluate the selected model on the given dataset, by invoking the graph workflow.
#     Save the dataframe with the questions and the generated cypher queries to the Results folder.
#     Return the dataframe
#     '''
#     # Get datasets 
#     if dataset =='sustain':
#         df = pd.read_excel('../data/text2cypher-sustaingraph/LLM Cypher Set.xlsx',sheet_name='Queries')
#         subfolder = 'SustainDataset'
#     elif dataset == 'benchmark2025':
#         df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
#         #update df only with rows with a database access
#         df = df[df['url'].notna()].reset_index(drop=True)
#         subfolder = 'BenchmarkDataset2025'
#     elif dataset == 'benchmark2024':
#         df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
#         #update df only with rows with a database access
#         df = df[df['url'].notna()].reset_index(drop=True)
#         subfolder = 'BenchmarkDataset2024'

#     output_dir = f"Results/{subfolder}/{model_name}_local/"
#     # Create the directory if it doesn't exist
#     os.makedirs(output_dir, exist_ok=True)
#     results_path_pkl = os.path.join(output_dir, f"{model_name}_local.pkl")
#     results_path_xlsx = os.path.join(output_dir, f"{model_name}_local.xlsx")
#     print(results_path_xlsx)

#     if os.path.exists(results_path_xlsx):
#         print("Result Excel file exists")
#         # Read Excel file
#         df_results = pd.read_excel(results_path_xlsx)
#         # print(df_results)
#         # Save it as pickle
#         df_results.to_pickle(results_path_pkl)
#         print("Converted Excel to pickle")
#         # Read the pickle
#         df_results = pd.read_pickle(results_path_pkl)
#         print("Read the pickle")
    
#     else:
#         print("Making result file")
#         # Run model on dataset
#         df_results = df.copy() # Copy of the df with questions and expected answers, to add the answers of the llm 
#         # Now define additional columns in df_results to store LLM results:
#         df_results['Generated_cypher'] = None 
#         df_results['answer'] = None
#         df_results['path'] = None           # Use None so we can store a list
#         df_results['output'] = None           # Use None so we can store a list
#         df_results['cypher_errors_history'] = None           # Use None so we can store a list
#         df_results['cypher_errors'] = None           # Use None so we can store a list
#         df_results['is_KG_Valid_correct'] = None           # Use None so we can store a list

#     config = {'configurable': {'model': model, 'model_name':model_name}}
#     print(f'Starting evaluation for {model_name} on {dataset} ...')

#     for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
#         print()
#         if pd.isna(row['Generated_Cypher']): 
#             question = row['Question'] #FIX: for benchmark2025, it's question, not Question
#             # #uncomment for benchmark
#             # database_name = row['database_name']

#             # if database_name == 'openstreetmap':
#             #     database_url = row['url']
#             #     database_name = row['database_name']
#             #     database_user = row['username']
#             #     database_password = row['password']

#             #     inputs = {
#             #     'question':question,
#             #     'database_url': database_url,
#             #     'database_name':database_name,
#             #     'database_user':database_user,
#             #     'database_password':database_password
#             #     }
#             #     torch.cuda.empty_cache() #clear gpu cache
#             #     # Run the graph
#             #     result =app.invoke(inputs, config =config) #,debug=False
#             #     # get_gpu_memory_in_gb()
#             #     df_results.at[i, 'Generated_Cypher'] = result['cypher_statement']
#             #     df_results.at[i, 'Output'] = str(result['database_records'])
#             #     torch.cuda.empty_cache()#clear gpu cache

#             #     df_results.to_pickle(results_path_pkl)
#             #     df_results.to_excel(results_path_xlsx, index=False)
#             #     # continue

#             # # if database_name == 'openstreetmap':
#             # #     df_results.at[i, 'Generated_Cypher'] = None
#             # #     df_results.at[i, 'Output'] = None
#             # #     continue

#             # # if row['question']=="Which three movies have the highest difference in revenue and budget?":
#             # #     df_results.at[i, 'Generated_Cypher'] = None
#             # #     df_results.at[i, 'Output'] = None
#             # #     continue             

#             # # Get credentials
#             # required_cols = {'database_name', 'url', 'username', 'password'}
#             # if required_cols.issubset(df_results.columns):
#             #     database_url = row['url']
#             #     database_name = row['database_name']
#             #     database_user = row['username']
#             #     database_password = row['password']
#             # else:
#             #     #its the sustain dataset
#             #     database_url = os.getenv('database_url')
#             #     database_name = os.getenv('database_name')
#             #     database_user = os.getenv('database_user')
#             #     database_password = os.getenv('database_password')
            
#             # # Configuration 
#             # inputs = {
#             #     'question':question,
#             #     'database_url': database_url,
#             #     'database_name':database_name,
#             #     'database_user':database_user,
#             #     'database_password':database_password
#             # }
#             # torch.cuda.empty_cache() #clear gpu cache
#             # # Run the graph
#             # result =app.invoke(inputs, config =config) #,debug=False
#             # # get_gpu_memory_in_gb()
#             # df_results.at[i, 'Generated_Cypher'] = result['cypher_statement']
#             # df_results.at[i, 'Output'] = str(result['database_records'])
#             # torch.cuda.empty_cache()#clear gpu cache

#             # df_results.to_pickle(results_path_pkl)
#             # df_results.to_excel(results_path_xlsx, index=False)
#         else:
#             print(f"Skipping row {i}, already computed.")
#     # # Save results
#     # output_dir = f"Results/{subfolder}/{model_name}/"
#     # # Create the directory if it doesn't exist
#     # os.makedirs(output_dir, exist_ok=True)
#     # # Save results
#     # df_results.to_pickle(os.path.join(output_dir, f"{model_name}.pkl"))
#     # df_results.to_excel(os.path.join(output_dir, f"{model_name}.xlsx"), index=False)# Set index=False to exclude the index column in Excel
#     return df_results

In [14]:
from Configuration.models import models #import models to evaluate
model_name = 'llama3_2_3b'
df_results = evaluate_model_and_save(model_name, models[model_name](), 'sustain')

WARNING! token is not default parameter.
                    token was transferred to model_kwargs.
                    Please make sure that token is what you intended.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Results/SustainDataset/llama3_2_3b_local/llama3_2_3b_local.xlsx
Result Excel file exists
Converted Excel to pickle
Read the pickle
Starting evaluation for llama3_2_3b on sustain ...


  0%|          | 0/52 [00:00<?, ?it/s]

Skipping row 0, already computed.
Skipping row 1, already computed.
Skipping row 2, already computed.
Skipping row 3, already computed.
Skipping row 4, already computed.
Skipping row 5, already computed.
Skipping row 6, already computed.
Skipping row 7, already computed.
Skipping row 8, already computed.
Skipping row 9, already computed.
Skipping row 10, already computed.
Skipping row 11, already computed.
Skipping row 12, already computed.
Skipping row 13, already computed.
Skipping row 14, already computed.
Skipping row 15, already computed.
Skipping row 16, already computed.
Skipping row 17, already computed.
Skipping row 18, already computed.
Skipping row 19, already computed.
Skipping row 20, already computed.
Skipping row 21, already computed.
Skipping row 22, already computed.
Skipping row 23, already computed.
Skipping row 24, already computed.
Skipping row 25, already computed.
Skipping row 26, already computed.
Skipping row 27, already computed.
Skipping row 28, already compu

In [ ]:
configuration_models = list(models.keys())
### Evaluate for each adataset the models that refer to this dataset (see models.py and guide.txt)
for dataset in ['sustain','benchamrk2025']: #benchamrk2024
    for model_name in configuration_models:
        print('Model name:', model_name)
        #Get model configuration
        model = models[model_name]()
        evaluate_model (model_name, model, dataset)

## Calculate metrics

In [ ]:
import sys
import pandas as pd
sys.path.append('../')
from utils.Evaluation.calculate_metrics import calculate_scores

model_name = 'llama3_2_3b_local'
df_results = pd.read_excel(f"Results/SustainDataset/{model_name}.xlsx")
#loop into folders with results 
df_results_metrics = calculate_scores(df_results)
#save the df_results_metrics inplace of the df_results
df_results_metrics

## Test

In [ ]:
question = "What is the percentage of the EGD documents that belong in the policy area Eliminating pollution?"
config = {'configurable': {'model': 'llama3_2_3b_local'}} #llama3_2_3b, openai
inputs = {
    'question':question,
    'database_url': os.getenv('SUSTAIN_DATABASE_URL'),
    'database_name':os.getenv('SUSTAIN_DATABASE_NAME'),
    'database_user':os.getenv('SUSTAIN_DATABASE_USER'),
    'database_password':os.getenv('SUSTAIN_DATABASE_PASSWORD')
}
result = app.invoke(inputs, config =config)
result

In [ ]:
exact_match = evaluate.load('exact_match')

print(exact_match.description)
print(exact_match.features)
exact_match.compute(references=['hello','hi'], predictions=['hello','hi!'])

In [ ]:
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])
clf_metrics.compute(predictions=[0, 1, 0], references=[0, 1, 1])

In [ ]:
evaluate-cli create "CyVer" --module_type "metric"